# Stage 2 — Mini Transformer for MNIST

Building a complete Transformer classifier on top of the attention module from Stage 1.

Three new components wrap the attention:
- **Residual connection**: `output = sublayer(X) + X` — allows gradients to flow directly to early layers
- **Layer Normalization**: normalises each token's features independently (no batch dependency)
- **Feed-Forward Network (FFN)**: two linear layers with GELU, expands then contracts the dimension

Together these form one **Transformer Block**, which is stacked `num_layers` times.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

## 1. Attention (from Stage 1)

Copied forward unchanged — later stages build on these same classes.

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    # [*, N, d_k] @ [*, d_k, N] -> [*, N, N]
    scores  = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    weights = torch.softmax(scores, dim=-1)  # [*, N, N]
    return weights @ V                       # [*, N, d_v]


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X):
        B, N, D = X.shape

        Q = self.W_Q(X)  # [B, N, D]
        K = self.W_K(X)  # [B, N, D]
        V = self.W_V(X)  # [B, N, D]

        # [B, N, D] -> [B, num_heads, N, d_k]
        Q = Q.reshape(B, N, self.num_heads, self.d_k).transpose(1, 2)
        K = K.reshape(B, N, self.num_heads, self.d_k).transpose(1, 2)
        V = V.reshape(B, N, self.num_heads, self.d_k).transpose(1, 2)

        out = scaled_dot_product_attention(Q, K, V)  # [B, num_heads, N, d_k]

        # [B, num_heads, N, d_k] -> [B, N, D]
        out = out.transpose(1, 2).reshape(B, N, D)
        return self.W_O(out)  # [B, N, D]

## 2. Transformer Block

One block = attention + FFN, each wrapped with a residual connection and Layer Norm.

This follows the **Post-LN** formulation from the original Transformer paper:
```
X = LayerNorm(X + Attention(X))
X = LayerNorm(X + FFN(X))
```

The FFN expands the dimension by 4× in the middle (`d_model -> 4*d_model -> d_model`),
giving the network capacity to learn non-linear feature interactions per token.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn  = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, X):
        # residual + attention, then normalise
        X = self.norm1(X + self.attn(X))  # [B, N, d_model]
        # residual + feed-forward, then normalise
        X = self.norm2(X + self.ffn(X))   # [B, N, d_model]
        return X

In [ ]:
B, N, d_model, num_heads = 2, 4, 256, 8
X = torch.randn(B, N, d_model)

block = TransformerBlock(d_model, num_heads)
out   = block(X)
print(out.shape)  # [2, 4, 256]

## 3. Mini Transformer Classifier

MNIST images are 28x28 pixels. We treat each **row** as a token: sequence length = 28, token dimension = 28.
No patch embedding needed — raw pixel rows go directly into the Transformer.

After stacking `num_layers` blocks, we collapse the sequence with **mean pooling** (average over tokens)
and pass the result through a linear classifier.

> Note: without positional encoding the model is permutation-equivariant — row order doesn't matter.
> We'll add positional encodings in Stage 3.

In [ ]:
class MiniTransformer(nn.Module):
    def __init__(self, d_model, num_heads, num_layers, num_classes):
        super().__init__()
        self.blocks     = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, X):
        # X: [B, 28, 28] — 28 tokens each of dimension 28
        for block in self.blocks:
            X = block(X)          # [B, 28, 28]
        X = X.mean(dim=1)         # [B, 28]  — mean pooling over tokens
        return self.classifier(X) # [B, num_classes]

In [ ]:
B = 4
X = torch.randn(B, 28, 28)

model = MiniTransformer(d_model=28, num_heads=4, num_layers=2, num_classes=10)
out   = model(X)
print(out.shape)  # [4, 10]

## 4. Training on MNIST

Input shape from DataLoader: `[B, 1, 28, 28]` (1 channel grayscale).
`squeeze(1)` removes the channel dimension to get `[B, 28, 28]` — our token sequence.

In [ ]:
D_MODEL     = 28   # one pixel per feature (= image width)
NUM_HEADS   = 4
NUM_LAYERS  = 2
NUM_CLASSES = 10
BATCH_SIZE  = 64
LR          = 1e-3
EPOCHS      = 5

In [ ]:
transform    = transforms.ToTensor()
train_set    = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_set     = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE)

In [ ]:
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = MiniTransformer(D_MODEL, NUM_HEADS, NUM_LAYERS, NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs   = imgs.squeeze(1).to(device)  # [B, 1, 28, 28] -> [B, 28, 28]
        labels = labels.to(device)

        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss / len(train_loader):.4f}")

In [ ]:
model.eval()
correct = total = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs   = imgs.squeeze(1).to(device)  # [B, 1, 28, 28] -> [B, 28, 28]
        labels = labels.to(device)
        preds  = model(imgs).argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

print(f"Test accuracy: {correct / total * 100:.2f}%")